# 05 — Transaction Engine (FULL OUTER JOIN)

Reproduces the SQL logic in pandas. For each consecutive quarter pair, `merge(how="outer")` on `(cusip, put_call)`:

| prev | curr | action |
|---|---|---|
| ∅ | ✓ | **NEW POSITION** |
| ✓ | ∅ | **FULL EXIT** |
| shares ↑ | | **BUY** |
| shares ↓ | | **SELL** |
| shares = | | HOLD |

**Why shares, not value:** market prices move every position's value every quarter — classifying on value would label *everything* a trade. Shares only change when the manager transacts. (Caveat: stock splits look like share changes; cross-check big anomalies.)

In [ ]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

In [ ]:
from src.transactions import build_transactions

tx = build_transactions()
tx.head(10)

In [ ]:
# Action counts per quarter
tx.pivot_table(index="quarter", columns="action", values="cusip",
               aggfunc="count", fill_value=0)

In [ ]:
# Biggest trades in the latest quarter
latest = tx["quarter"].max()
(tx[(tx["quarter"] == latest) & (tx["action"] != "HOLD")]
   .reindex(tx[tx["quarter"] == latest]["value_change_usd"].abs().sort_values(ascending=False).index)
   .dropna(subset=["action"])
   [["issuer", "action", "share_change", "value_change_usd"]]
   .head(15))